In [29]:
import numpy as np
import pandas as pd
import os

import cv2
from skimage.feature import hog
from skimage import exposure
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

In [30]:
df = pd.read_csv('image_metadata.csv')
df

,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
0,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,0,P. aeruginosa
1,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,1,P. aeruginosa
2,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,2,P. aeruginosa
3,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,3,P. aeruginosa
4,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,4,P. aeruginosa
...,...,...,...,...,...,...,...,...
106838,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,25,E. coli
106839,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,26,E. coli
106840,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,27,E. coli
106841,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,28,E. coli


In [31]:
filtered_df = df[(df['species_label'] != 'Unknown') & #only working with known species
                 (df['treated'] == 'Untreated') &  # removed treated antibiotic samples since the later timepoint images are harder to use
                 (df['timepoint'] == 15)] # looking at frame 15 for the 30 minute point
filtered_df

,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
15,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,15,P. aeruginosa
49,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,30,43-08,Untreated,15,P. aeruginosa
83,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,40,43-18,Untreated,15,P. aeruginosa
117,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,41,43-19,Untreated,15,P. aeruginosa
151,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,102,7,123-07,Untreated,15,P. aeruginosa
...,...,...,...,...,...,...,...,...
100708,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,179,29,5647-12,Untreated,15,K. pneumoniae
100738,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,180,23,5715-06,Untreated,15,P. aeruginosa
100768,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,180,25,5715-08,Untreated,15,K. pneumoniae
100798,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,180,29,5715-12,Untreated,15,K. pneumoniae


In [32]:
filtered_df['species_label'].value_counts()

species_label
K. pneumoniae    1271
E. coli           582
E. faecalis       371
P. aeruginosa     250
Name: count, dtype: int64

In [33]:
# n = 250
# filtered_df_sampled = filtered_df.groupby('species_label').apply(lambda x: x.sample(n=n, random_state=18)).reset_index(drop=True)
# filtered_df_sampled #adjusted to 250 samples per species to account for majority class imbalance
filter_df_sampled = filtered_df

In [34]:
def load_image(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (128, 128))  # Resize
    return image

In [35]:
def extract_hog_features(image):
    features, hog_image = hog(image, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
    #hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))
    return features

In [36]:
def load_dataset(image_paths, labels):
    X = []
    y = []
    for img_path, label in zip(image_paths, labels):
        image = load_image(img_path)
        #features = extract_hog_features(image)
        features = extract_features(image)
        X.append(features)
        y.append(label)
    return np.array(X), np.array(y)

In [37]:
def get_paths_and_labels(filtered_df_sampled):    
    img_paths = []
    labels = []
    for _, row in filtered_df_sampled.iterrows():
        img_path = row['file_path']  # Path to image file
        species_label = row['species_label']  # Target variable
        
        img_paths.append(img_path)
        labels.append(species_label)
    return img_paths, labels
img_paths, labels = get_paths_and_labels(filtered_df_sampled)

In [38]:
species_map = {'E. faecalis': 0, 'K. pneumoniae': 1, 'E. coli': 2, 'P. aeruginosa': 3}
numerical_labels = [species_map[label] for label in labels]

X, y = load_dataset(img_paths, numerical_labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23, stratify=y)


In [39]:
print(f"Training set class distribution: {pd.Series(y_train).value_counts()}")
print(f"Test set class distribution: {pd.Series(y_test).value_counts()}")

Training set class distribution: 1    200
0    200
2    200
3    200
Name: count, dtype: int64
Test set class distribution: 1    50
2    50
3    50
0    50
Name: count, dtype: int64


In [40]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [41]:
clf = SVC(kernel='linear')
clf.fit(X_train, y_train)

SVC(kernel='linear')

In [42]:
y_pred = clf.predict(X_test)

In [43]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      0.92      0.96        50
           2       0.83      0.96      0.89        50
           3       0.96      0.88      0.92        50

    accuracy                           0.94       200
   macro avg       0.95      0.94      0.94       200
weighted avg       0.95      0.94      0.94       200



In [44]:
def extract_features(image): #hog, canny, and contours
    hog_features, hog_image = hog(image, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
    
    edges = cv2.Canny(image, 100, 200)
    edge_features = edges.flatten()
    
    ret, thresh = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    contour_features = []
    
    max_contours = 5 # Limit to first 5 contours
    for contour in contours[:max_contours]:  
        moments = cv2.moments(contour)
        hu_moments = cv2.HuMoments(moments).flatten()
        contour_features.extend(hu_moments)
    
    # Pad if fewer than max_contours are found
    while len(contour_features) < max_contours * 7:
        contour_features.append(0.0)

    contour_features =  np.array(contour_features)
    
    features = np.hstack([hog_features, edge_features, contour_features])
    return features